In [ ]:
import msgpack
import numpy as np
import base64
import requests
import librosa
from IPython.display import Audio
import json

In [ ]:
from sjpy.audio import ndarray_to_mp4_bytes, array_to_s16le, segment

In [ ]:
SAMPLE_RATE = 16000
# BASE_URL = "https://api.sangjeong.com:8080"
BASE_URL = "http://localhost:8000"

# 오디오 로드 및 세그멘팅

In [ ]:
original_audio, sr = librosa.load(".storage/boda.wav", sr=SAMPLE_RATE)
# original_audio, sr = librosa.load(".data/news_with_english.wav", sr=SAMPLE_RATE)

In [ ]:
segments = segment(original_audio, 4000, 30, 300)

# 오디오 출력해보기

In [ ]:
idx = 0

In [ ]:
audio = segments[idx]
idx += 1
Audio(audio, rate=SAMPLE_RATE)

# 유틸

In [ ]:
def dump_using_json(data:dict):
    text = json.dumps(data).encode("utf-8")
    length = len(text).to_bytes(4, byteorder="big")
    return length + text

def dump_using_msgpack(data:dict):
    text = msgpack.packb(data)
    length = len(text).to_bytes(4, byteorder="big")
    return length + text

# /diarization

In [ ]:
# 예시 음성 데이터의 일부
refer_dict = {
    "0": [(148061, 339589), (338760, 436145), (433746, 517335)],
    "1": [(1013274, 1068548), (1071733, 1209101), (1209858, 1344749)],
    "2": [(2563542, 2662059), (4373060, 4438192), (4570430, 4625790)],
    "3": [(880324, 938541), (7042377, 7096457), (7106339, 7156845)],
    "4": [(944284, 1009464), (1802918, 1921616), (1921626, 2053013)],
}

#### /diarization/embed_stream

In [ ]:
embedding_refer_dict = {"0": [], "1": [], "2": [], "3": [], "4": []}
for idx, ts in refer_dict.items():
    for s in ts:
        audio = original_audio[s[0] : s[1]]
        bt = ndarray_to_mp4_bytes(audio)
        res = requests.post(
            BASE_URL + "/diarization/embed_stream",
            data=bt,
            headers={"Content-Type": "application/octet-stream"},
        )
        embedding_refer_dict[idx].append(res.json()["embedding"])

In [ ]:
# 임베딩 데이터 확인
for k, v in embedding_refer_dict.items():
    print(f"Speaker {k}: {len(v)} embeddings")
    for i, emb in enumerate(v):
        print(f"\tEmbedding {i}: length {len(emb)}")
        bytes_ = base64.b64decode(emb)
        print(f"\tBytes length: {len(bytes_)}")
        embedding = np.frombuffer(bytes_, dtype=np.float32)
        print(f"\tNumpy array shape: {embedding.shape}")

#### /diarization/embed_file

In [ ]:
embedding_refer_dict = {"0": [], "1": [], "2": [], "3": [], "4": []}
for idx, ts in refer_dict.items():
    for s in ts:
        audio = original_audio[s[0] : s[1]]
        bt = ndarray_to_mp4_bytes(audio)
        res = requests.post(
            BASE_URL + "/diarization/embed_file",
            files={"audio": ("audio.bin", bt, "application/octet-stream")},
        )
        embedding_refer_dict[idx].append(res.json()["embedding"])

In [ ]:
# 임베딩 데이터 확인
for k, v in embedding_refer_dict.items():
    print(f"Speaker {k}: {len(v)} embeddings")
    for i, emb in enumerate(v):
        print(f"\tEmbedding {i}: length {len(emb)}")
        bytes_ = base64.b64decode(emb)
        print(f"\tBytes length: {len(bytes_)}")
        embedding = np.frombuffer(bytes_, dtype=np.float32)
        print(f"\tNumpy array shape: {embedding.shape}")

#### /diarization/refer_stream

In [ ]:
keys = list(embedding_refer_dict.keys())
bts = b''
for k in keys:
    for emb in embedding_refer_dict[k]:
        bts += base64.b64decode(emb)

print(f"Total bytes length: {len(bts)}")

res = requests.post(
    BASE_URL + "/diarization/refer_stream",
    params = {
        "group_id": "0",
        "user_id": "0", # 소켓에서는 필요 없다. http 통신에서는 어쩔 수 없이 넣어줘야 한다.
        "user_ids": keys,
        "counts": [len(embedding_refer_dict[k]) for k in keys],
    },
    data=bts,
)
print(res.status_code)

#### /diarization/refer_file

In [ ]:
res = requests.post(
    BASE_URL + "/diarization/refer_file",
    params={
        "group_id": "0",
        "user_id": "0",
        "user_ids": keys,
        "counts": [len(embedding_refer_dict[k]) for k in keys]
    },
    files={"refers": ("audio.bin", bts, "application/octet-stream")},
)
print(res.status_code)

#### /diarization/diarize_stream

In [ ]:
def print_sentences(res_json):
    group_id, completed, candidate = res_json["group_id"], res_json["completed"], res_json["candidate"]
    if completed or candidate:
        print(f"Group ID: {group_id}")
        print(f"completed:")
        for sent in completed:
            print(f'\t{sent["order"]} | {sent["user_id"]}: {sent["text"]}')
        print(f"candidate:")
        for sent in candidate:
            print(f'\t{sent["order"]} | {sent["user_id"]} (candidate): {sent["text"]}')

In [ ]:
pcm_audio = array_to_s16le(original_audio)
pcm_segments = segment(pcm_audio, 4000, 30, 300)

In [ ]:
# 임베딩 벡터 힌트를 제공하면, 같은 그룹 내에서 화자분리를 실행한다.
# 임베딩 벡터가 없는 사용자는, 화자분리를 동작하지 않고, 아이디와 같은 화자로 간주한다.
bt = pcm_segments[0].tobytes()
res = requests.post(
    BASE_URL + "/diarization/diarize_stream",
    params={"group_id": "0", "user_id": "0", "sc_offset": 80000}, # sample rate offset을 넣으려면 sc_offset 파라미터 추가하면 된다.
    data=bt,
    headers={"Content-Type": "application/octet-stream"},
)
print_sentences(res.json())

for seg in pcm_segments[1:100]:
    res = requests.post(
        BASE_URL + "/diarization/diarize_stream",
        params={"group_id": "0", "user_id": "0"}, # sample rate offset을 넣으려면 sc_offset 파라미터 추가하면 된다.
        data=seg.tobytes(),
        headers={"Content-Type": "application/octet-stream"},
    )
    print_sentences(res.json())

#### /diarization/diarize_file

In [ ]:
for seg in segments[100:200]:
    res = requests.post(
        BASE_URL + "/diarization/diarize_file",
        params={"group_id": "0", "user_id": "0"},
        files={"audio": ("audio.opus", seg.tobytes(), "application/octet-stream")},
    )
    print_sentences(res.json())

res = requests.post(
    BASE_URL + "/diarization/diarize_file",
    params={"group_id": "0", "user_id": "0"},
    files={"audio": ("audio.opus", b'', "application/octet-stream")},
)
print_sentences(res.json())

#### /llm/metadata

In [ ]:
res = requests.get(
    BASE_URL + "/llm/metadata",
    params={"group_id": "0"},
)

In [ ]:
res = requests.get(
    BASE_URL + "/llm/context",
    params={"group_id": "0"},
)
result = res.json()
print(result)

In [ ]:
res = requests.get(
    BASE_URL + "/llm/context_done",
    params={"group_id": "0"},
)
result = res.json()
print(result)